# 셀1. 경로

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/bigcontest-2026")
OUT = BASE / "data/processed"

In [ ]:
from pathlib import Path

print(list(Path("/content/drive").iterdir()))

In [ ]:
from pathlib import Path

base = Path("/content/drive/.shortcut-targets-by-id")

for p in base.iterdir():
    print(p)

셀 2. 외지인/지역민 합쳐서 "날짜 × 지역 × 업종"으로

In [ ]:
av = pd.read_csv(OUT / "analysis_visitor.csv", parse_dates=["date"])

KEYS = ["date", "region", "ind_grp"]
DAY = ["dow", "weekend", "holiday", "chuseok", "month",
       "tmax", "feels_max", "rain", "heat", "heat_streak", "tmax_lag1"]   # 날짜별로 같은 값

df = (av.groupby(KEYS, as_index=False)
        .agg(sales=("sales", "sum"), **{c: (c, "first") for c in DAY}))

print(df.shape)                                                    # 약 4,784행 예상
print(df.groupby(["region", "ind_grp"])["date"].nunique().unique())  # [184] 이어야 함

# 셀 3. 맞힐 값 + 지난주 매출

In [ ]:
df = df.sort_values(KEYS).reset_index(drop=True)
df["y"] = np.log(df["sales"])                          # 맞힐 값

g = df.groupby(["region", "ind_grp"])["sales"]
for k in [7, 14, 21, 28]:
    df[f"lag{k}"] = g.shift(k)                         # k일 전 = 같은 요일
df["lag7_mean4"] = df[["lag7", "lag14", "lag21", "lag28"]].mean(axis=1)   # 있는 주만 평균
df = df.drop(columns=["lag14", "lag21", "lag28"])

# 셀 4. 날씨 파생 변수 + 주 번호

In [ ]:
w = df[["date", "region", "rain"]].drop_duplicates().sort_values(["region", "date"])
w["rain_lag1"] = w.groupby("region")["rain"].shift(1)   # 전날 비
df = df.merge(w[["date", "region", "rain_lag1"]], on=["date", "region"])

df["rainy"] = (df["rain"] >= 1).astype(int)            # 비 온 날
df["heavy"] = (df["rain"] >= 30).astype(int)           # 호우

# 주 번호: 6/30(월)부터 0, 1, 2 … (월요일 시작)
df["week"] = ((df["date"] - pd.Timestamp("2025-06-30")).dt.days // 7).astype(int)

# 셀 5. 정리 + 저장

In [ ]:
before = len(df)
df = df.dropna(subset=["lag7", "tmax_lag1", "rain_lag1"]).reset_index(drop=True)   # 첫 주 제외
print(f"첫 주 제외: {before:,} → {len(df):,}행")

COLS = ["date", "region", "ind_grp", "week", "sales", "y",
        "dow", "weekend", "holiday", "chuseok", "month",
        "lag7", "lag7_mean4",
        "tmax", "feels_max", "rain", "rainy", "heavy", "heat", "heat_streak",
        "tmax_lag1", "rain_lag1"]
df = df[COLS]
df.to_csv(OUT / "model_input.csv", index=False, encoding="utf-8-sig")

# 점검
print("결측:", df.isna().sum().sum())
print("기간:", df["date"].min().date(), "~", df["date"].max().date(), "| 주 번호:", df["week"].min(), "~", df["week"].max())
day = df.drop_duplicates(["date", "region"])
print(day.groupby("region")[["heat", "rainy", "heavy", "holiday", "chuseok"]].sum())
print("y와 log(lag7) 상관:", round(np.corrcoef(df["y"], np.log(df["lag7"]))[0, 1], 3))

In [ ]:
# ============================================================
# 시계열 Walk-forward 검증
# ① 기준 모델
# ② LightGBM 날씨 없음
# ③ LightGBM 날씨 포함
# ============================================================

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import mean_absolute_percentage_error


# ------------------------------------------------------------
# 1. 데이터 불러오기
# ------------------------------------------------------------

model_df = pd.read_csv(
    OUT / "model_input.csv",
    parse_dates=["date"]
)

model_df = model_df.sort_values("date").reset_index(drop=True)

print("데이터 크기:", model_df.shape)
print(
    "기간:",
    model_df["date"].min().date(),
    "~",
    model_df["date"].max().date()
)


# ------------------------------------------------------------
# 2. 평가 함수
# ------------------------------------------------------------

def mape_percent(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # 실제 매출이 0인 경우 제외
    mask = y_true > 0

    if mask.sum() == 0:
        return np.nan

    return mean_absolute_percentage_error(
        y_true[mask],
        y_pred[mask]
    ) * 100


# ------------------------------------------------------------
# 3. 사용할 변수
# ------------------------------------------------------------

TARGET = "y"

# 날씨 없는 모델
BASE_FEATURES = [
    "region",
    "ind_grp",
    "week",
    "dow",
    "weekend",
    "holiday",
    "chuseok",
    "month",
    "lag7",
    "lag7_mean4"
]

# 날씨 변수
WEATHER_FEATURES = [
    "tmax",
    "feels_max",
    "rain",
    "rainy",
    "heavy",
    "heat",
    "heat_streak",
    "tmax_lag1",
    "rain_lag1"
]


# ------------------------------------------------------------
# 4. 범주형 변수 처리
# ------------------------------------------------------------

def make_features(train_df, test_df, features):

    X_train = train_df[features].copy()
    X_test = test_df[features].copy()

    categorical = ["region", "ind_grp"]

    all_data = pd.concat(
        [X_train, X_test],
        axis=0
    )

    all_data = pd.get_dummies(
        all_data,
        columns=categorical,
        dtype=int
    )

    X_train = all_data.iloc[:len(X_train)].copy()
    X_test = all_data.iloc[len(X_train):].copy()

    return X_train, X_test


# ------------------------------------------------------------
# 5. LightGBM 학습
# ------------------------------------------------------------

def train_lgbm(train_df, test_df, features):

    X_train, X_test = make_features(
        train_df,
        test_df,
        features
    )

    y_train = train_df[TARGET]

    model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=-1
    )

    model.fit(
        X_train,
        y_train
    )

    pred_log = model.predict(X_test)

    # log1p 변환을 되돌림
    pred_sales = np.expm1(pred_log)

    # 음수 예측 방지
    pred_sales = np.maximum(pred_sales, 0)

    return model, pred_sales


# ============================================================
# 6. Walk-forward 검증
# ============================================================

# 월별로 검증
model_df["year_month"] = model_df["date"].dt.to_period("M")

months = sorted(model_df["year_month"].unique())

# 첫 달은 학습 데이터가 부족하므로 제외
test_months = months[1:]

all_predictions = []

print("\n")
print("=" * 80)
print("Walk-forward 검증")
print("=" * 80)


for test_month in test_months:

    # 해당 월 이전 데이터로 학습
    train_df = model_df[
        model_df["year_month"] < test_month
    ].copy()

    # 해당 월 데이터로 검증
    test_df = model_df[
        model_df["year_month"] == test_month
    ].copy()

    # 학습 데이터가 너무 적으면 건너뜀
    if len(train_df) < 500:
        continue

    print(
        f"{test_month} | "
        f"train={len(train_df):,} | "
        f"test={len(test_df):,}"
    )

    # --------------------------------------------------------
    # ① 기준 모델
    # --------------------------------------------------------

    pred_baseline = test_df["lag7"].values

    # --------------------------------------------------------
    # ② LightGBM 날씨 없음
    # --------------------------------------------------------

    _, pred_no_weather = train_lgbm(
        train_df,
        test_df,
        BASE_FEATURES
    )

    # --------------------------------------------------------
    # ③ LightGBM 날씨 포함
    # --------------------------------------------------------

    _, pred_weather = train_lgbm(
        train_df,
        test_df,
        BASE_FEATURES + WEATHER_FEATURES
    )

    # 예측 결과 저장
    result = test_df[
        [
            "date",
            "region",
            "ind_grp",
            "sales",
            "heat",
            "rainy",
            "heavy"
        ]
    ].copy()

    result["pred_baseline"] = pred_baseline
    result["pred_lgbm_no_weather"] = pred_no_weather
    result["pred_lgbm_weather"] = pred_weather

    all_predictions.append(result)


# ------------------------------------------------------------
# 7. 모든 검증 결과 합치기
# ------------------------------------------------------------

test = pd.concat(
    all_predictions,
    ignore_index=True
)

print("\n검증 데이터:", len(test), "행")
print(
    "검증기간:",
    test["date"].min().date(),
    "~",
    test["date"].max().date()
)


# ============================================================
# 8. 상황별 평가
# ============================================================

conditions = {

    "전체":
        np.ones(len(test), dtype=bool),

    "폭염일":
        test["heat"] == 1,

    "비 온 날":
        test["rainy"] == 1,

    "호우일":
        test["heavy"] == 1,

    "강남 호우일":
        (test["region"] == "강남") &
        (test["heavy"] == 1),

    "춘천 호우일":
        (test["region"] == "춘천") &
        (test["heavy"] == 1),

    "춘천 야외레저 호우일":
        (test["region"] == "춘천") &
        (test["ind_grp"] == "야외레저") &
        (test["heavy"] == 1)
}


# ============================================================
# 9. 결과표
# ============================================================

results = []

for condition_name, mask in conditions.items():

    sub = test.loc[mask].copy()

    results.append({

        "조건": condition_name,

        "표본수": len(sub),

        "① 기준": round(
            mape_percent(
                sub["sales"],
                sub["pred_baseline"]
            ),
            2
        ) if len(sub) > 0 else np.nan,

        "② LightGBM 날씨 없음": round(
            mape_percent(
                sub["sales"],
                sub["pred_lgbm_no_weather"]
            ),
            2
        ) if len(sub) > 0 else np.nan,

        "③ LightGBM 날씨 포함": round(
            mape_percent(
                sub["sales"],
                sub["pred_lgbm_weather"]
            ),
            2
        ) if len(sub) > 0 else np.nan
    })


results_df = pd.DataFrame(results)


# ============================================================
# 10. 날씨 포함 모델 개선율
# ============================================================

results_df["날씨 포함 개선율(%)"] = (
    (
        results_df["② LightGBM 날씨 없음"]
        -
        results_df["③ LightGBM 날씨 포함"]
    )
    /
    results_df["② LightGBM 날씨 없음"]
    * 100
).round(2)


# ============================================================
# 11. 최종 결과
# ============================================================

print("\n")
print("=" * 80)
print("최종 모델 성능 비교")
print("=" * 80)

display(results_df)

In [ ]:
# ============================================================
# MAE + WAPE 추가 평가
# ============================================================

from sklearn.metrics import mean_absolute_error


# ------------------------------------------------------------
# 1. 평가 함수
# ------------------------------------------------------------

def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)


def wape(y_true, y_pred):
    """
    WAPE = Σ|실제-예측| / Σ|실제| × 100
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    denominator = np.sum(np.abs(y_true))

    if denominator == 0:
        return np.nan

    return (
        np.sum(np.abs(y_true - y_pred))
        / denominator
        * 100
    )


# ------------------------------------------------------------
# 2. 조건별 평가
# ------------------------------------------------------------

metrics_results = []

for condition_name, mask in conditions.items():

    sub = test.loc[mask].copy()

    if len(sub) == 0:
        continue

    # 실제 매출
    y = sub["sales"]

    metrics_results.append({

        "조건": condition_name,

        "표본수": len(sub),

        # =========================
        # ① 기준
        # =========================
        "① 기준 MAPE": round(
            mape_percent(
                y,
                sub["pred_baseline"]
            ),
            2
        ),

        "① 기준 MAE": round(
            mae(
                y,
                sub["pred_baseline"]
            ),
            0
        ),

        "① 기준 WAPE": round(
            wape(
                y,
                sub["pred_baseline"]
            ),
            2
        ),

        # =========================
        # ② LightGBM 날씨 없음
        # =========================
        "② 날씨 없음 MAPE": round(
            mape_percent(
                y,
                sub["pred_lgbm_no_weather"]
            ),
            2
        ),

        "② 날씨 없음 MAE": round(
            mae(
                y,
                sub["pred_lgbm_no_weather"]
            ),
            0
        ),

        "② 날씨 없음 WAPE": round(
            wape(
                y,
                sub["pred_lgbm_no_weather"]
            ),
            2
        ),

        # =========================
        # ③ LightGBM 날씨 포함
        # =========================
        "③ 날씨 포함 MAPE": round(
            mape_percent(
                y,
                sub["pred_lgbm_weather"]
            ),
            2
        ),

        "③ 날씨 포함 MAE": round(
            mae(
                y,
                sub["pred_lgbm_weather"]
            ),
            0
        ),

        "③ 날씨 포함 WAPE": round(
            wape(
                y,
                sub["pred_lgbm_weather"]
            ),
            2
        )
    })


metrics_df = pd.DataFrame(metrics_results)


# ============================================================
# 3. 결과 출력
# ============================================================

print("=" * 100)
print("MAPE + MAE + WAPE 종합 평가")
print("=" * 100)

display(metrics_df)

In [ ]:
# ============================================================
# 구간별 ①②③ WAPE 비교표
# ============================================================

wape_table = metrics_df[
    [
        "조건",
        "표본수",
        "① 기준 WAPE",
        "② 날씨 없음 WAPE",
        "③ 날씨 포함 WAPE"
    ]
].copy()

# ② → ③ 날씨 포함 개선율
wape_table["날씨 포함 개선율(%)"] = (
    1
    - wape_table["③ 날씨 포함 WAPE"]
    / wape_table["② 날씨 없음 WAPE"]
) * 100

wape_table["날씨 포함 개선율(%)"] = (
    wape_table["날씨 포함 개선율(%)"].round(2)
)

display(wape_table)
wape_table.to_csv(BASE / "outputs/tables/07_performance.csv", index=False, encoding="utf-8-sig")

# 비가 안 왔을 때 vs 45mm 왔을 때 매출이 얼마나 차이 날까?

In [ ]:
import lightgbm as lgb

df = pd.read_csv(OUT / "model_input.csv", parse_dates=["date"])
for c in ["region", "ind_grp"]:
    df[c] = df[c].astype("category")

BASE_FEATS = ["dow","weekend","holiday","chuseok","month","lag7","lag7_mean4","region","ind_grp"]
WX_FEATS = ["tmax","feels_max","rain","rainy","heavy","heat","heat_streak","tmax_lag1","rain_lag1"]
FEATS = BASE_FEATS + WX_FEATS
PARAMS = dict(objective="regression", learning_rate=0.05, n_estimators=300, num_leaves=15,
              max_depth=4, min_child_samples=5, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.8, random_state=42, verbose=-1)

# 전체 데이터로 최종 모델 학습
final_wx = lgb.LGBMRegressor(**PARAMS).fit(df[FEATS], df["y"])

def predict_with_rain(d, mm):
    s = d.copy()
    s["rain"] = mm
    s["rainy"] = int(mm >= 1)
    s["heavy"] = int(mm >= 30)
    return np.exp(final_wx.predict(s[FEATS]))

# 실제로 비가 안 온 여름날(7~9월)만 사용
base = df[(df["month"].between(7, 9)) & (df["rain"] < 1)]

rows = []
for (r, g), d in base.groupby(["region", "ind_grp"], observed=True):
    dry = predict_with_rain(d, 0).sum()
    row = {"지역": r, "업종": g}
    for mm in [10, 30, 45]:
        row[f"{mm}mm"] = round((predict_with_rain(d, mm).sum() / dry - 1) * 100, 1)
    rows.append(row)

scen = pd.DataFrame(rows).sort_values("45mm")
print(scen.to_string(index=False))

TAB = OUT.parents[1] / "outputs" / "tables"   # data/processed → 프로젝트 폴더 → outputs/tables
TAB.mkdir(parents=True, exist_ok=True)
scen.to_csv(TAB / "07_scenario.csv", index=False, encoding="utf-8-sig")

# ① 구간별 오차 막대그래프

In [ ]:
# ============================================================
# 한글 폰트 강제 등록
# ============================================================

!apt-get update -qq
!apt-get install -y fonts-nanum -qq

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

# 폰트 파일 존재 확인
print("폰트 파일 존재:", os.path.exists(font_path))

# Matplotlib에 폰트 직접 등록
fm.fontManager.addfont(font_path)

# 폰트 이름 확인
font_prop = fm.FontProperties(fname=font_path)
font_name = font_prop.get_name()

print("등록된 폰트 이름:", font_name)

# Matplotlib 기본 폰트로 지정
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print("한글 폰트 설정 완료!")

In [ ]:
# ============================================================
# ① 구간별 모델 WAPE 비교
# ============================================================

plot_df = wape_table.copy()

conditions_order = [
    "전체",
    "폭염일",
    "비 온 날",
    "호우일",
    "강남 호우일",
    "춘천 호우일",
    "춘천 야외레저 호우일"
]

plot_df = (
    plot_df
    .set_index("조건")
    .loc[conditions_order]
    .reset_index()
)

x = np.arange(len(plot_df))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 7))

ax.bar(
    x - width,
    plot_df["① 기준 WAPE"],
    width,
    label="① 기준"
)

ax.bar(
    x,
    plot_df["② 날씨 없음 WAPE"],
    width,
    label="② 날씨 없음"
)

ax.bar(
    x + width,
    plot_df["③ 날씨 포함 WAPE"],
    width,
    label="③ 날씨 포함"
)

ax.set_xticks(x)
ax.set_xticklabels(
    plot_df["조건"],
    rotation=20,
    ha="right"
)

ax.set_ylabel("WAPE (%)")
ax.set_title("기상 상황별 모델 예측 오차 비교")

ax.legend()

plt.tight_layout()
plt.savefig(BASE / "outputs/figures/07_wape_by_condition.png", dpi=200, bbox_inches="tight")
plt.show()

# ② SHAP 변수 중요도

②-1. SHAP 계산

In [ ]:
# ============================================================
# SHAP에 사용할 실제 변수명 확인
# ============================================================

print("===== 모델 변수 =====")

model_names = [
    name for name in list(globals().keys())
    if "model" in name.lower()
]

for name in model_names:
    try:
        print(name, "→", type(globals()[name]).__name__)
    except:
        pass


print("\n===== X 데이터 변수 =====")

x_names = [
    name for name in list(globals().keys())
    if name.startswith("X")
]

for name in x_names:
    try:
        obj = globals()[name]
        print(
            name,
            "→",
            type(obj).__name__,
            "shape =", getattr(obj, "shape", "없음")
        )
    except:
        pass

In [ ]:
print("===== train_df 컬럼 =====")
for i, col in enumerate(train_df.columns, 1):
    print(i, ":", col)

print("\n===== model_df 컬럼 =====")
for i, col in enumerate(model_df.columns, 1):
    print(i, ":", col)

In [ ]:
# ============================================================
# ②-1. 날씨 포함 모델(model_weather) 생성
# ============================================================

import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor

# ------------------------------------------------------------
# 1. 사용할 기본 변수
# ------------------------------------------------------------

NUMERIC_FEATURES = [
    "week",
    "dow",
    "weekend",
    "holiday",
    "chuseok",
    "month",
    "lag7",
    "lag7_mean4",
    "tmax",
    "feels_max",
    "rain",
    "rainy",
    "heavy",
    "heat",
    "heat_streak",
    "tmax_lag1",
    "rain_lag1"
]

# ------------------------------------------------------------
# 2. region / ind_grp 더미변수 생성
#    train + test를 합쳐서 동일한 컬럼 구조를 만듦
# ------------------------------------------------------------

temp_train = train_df.copy()
temp_test = test_df.copy()

combined = pd.concat(
    [
        temp_train[["region", "ind_grp"]],
        temp_test[["region", "ind_grp"]]
    ],
    axis=0
)

combined = pd.get_dummies(
    combined,
    columns=["region", "ind_grp"],
    dtype=int
)

# 다시 train / test로 분리
train_cat = combined.iloc[:len(temp_train)].reset_index(drop=True)
test_cat = combined.iloc[len(temp_train):].reset_index(drop=True)

# ------------------------------------------------------------
# 3. 숫자 변수 + 더미변수 합치기
# ------------------------------------------------------------

X_train_weather = pd.concat(
    [
        temp_train[NUMERIC_FEATURES].reset_index(drop=True),
        train_cat.reset_index(drop=True)
    ],
    axis=1
)

X_test_weather = pd.concat(
    [
        temp_test[NUMERIC_FEATURES].reset_index(drop=True),
        test_cat.reset_index(drop=True)
    ],
    axis=1
)

# ------------------------------------------------------------
# 4. 혹시 중복 컬럼이 있으면 제거
# ------------------------------------------------------------

X_train_weather = X_train_weather.loc[
    :, ~X_train_weather.columns.duplicated()
]

X_test_weather = X_test_weather.loc[
    :, ~X_test_weather.columns.duplicated()
]

# train / test 컬럼 완전히 동일하게 맞추기
X_test_weather = X_test_weather.reindex(
    columns=X_train_weather.columns,
    fill_value=0
)

# ------------------------------------------------------------
# 5. 최종 변수 확인
# ------------------------------------------------------------

print("===== X_train_weather =====")
print("행:", len(X_train_weather))
print("변수 개수:", X_train_weather.shape[1])
print()

for i, col in enumerate(X_train_weather.columns, 1):
    print(i, ":", col)

# ------------------------------------------------------------
# 6. 모델 생성
# ------------------------------------------------------------

model_weather = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

# y를 예측
model_weather.fit(
    X_train_weather,
    train_df["y"]
)

print()
print("======================================")
print("model_weather 생성 완료!")
print("변수 개수:", len(model_weather.feature_name_))
print("======================================")

#②-2. SHAP 변수 중요도 그래프

In [ ]:
# ============================================================
# SHAP 변수 중요도 - 전체 기간 + 춘천 호우일
# ============================================================

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

# ★ 추가
FIG = BASE / "outputs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. 모델에 사용할 변수
# ------------------------------------------------------------

FEATS_WX = BASE_FEATURES + WEATHER_FEATURES

X_all = pd.get_dummies(
    model_df[FEATS_WX],
    columns=["region", "ind_grp"],
    dtype=int
)

# ------------------------------------------------------------
# 2. 모델 학습
# ------------------------------------------------------------

model_weather = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1
)

model_weather.fit(
    X_all,
    model_df["y"]
)

print("모델 학습 완료")
print("변수 개수:", X_all.shape[1])

# ------------------------------------------------------------
# 3. SHAP 계산
# ------------------------------------------------------------

explainer = shap.TreeExplainer(model_weather)
shap_values = explainer.shap_values(X_all)

imp = pd.DataFrame(
    np.abs(shap_values),
    columns=X_all.columns
)

# ------------------------------------------------------------
# 4. SHAP 그래프 함수
# ------------------------------------------------------------

def plot_shap(mask, title, filename):

    s = imp[mask].mean()
    s = s.sort_values(ascending=False)

    weather_cols = [
        c for c in s.index
        if c in WEATHER_FEATURES
    ]

    weather_share = (
        s[weather_cols].sum() / s.sum() * 100
        if len(weather_cols) > 0
        else 0
    )

    top = s.head(12).sort_values()

    plt.figure(figsize=(9, 6))

    plt.barh(
        top.index,
        top.values
    )

    plt.xlabel("평균 |SHAP 값|")
    plt.title(
        f"{title}\n"
        f"날씨 변수 비중: {weather_share:.1f}%"
    )

    plt.tight_layout()

    plt.savefig(
        FIG / filename,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()

    print(f"{title}")
    print(f"날씨 변수 비중: {weather_share:.1f}%")
    print()

    return s

# ------------------------------------------------------------
# 5. 전체 기간 SHAP
# ------------------------------------------------------------

s_all = plot_shap(
    np.ones(len(model_df), dtype=bool),
    "전체 기간 변수 중요도",
    "07_shap_all.png"
)

# ------------------------------------------------------------
# 6. 춘천 호우일 SHAP
# ------------------------------------------------------------

heavy_mask = (
    (model_df["region"] == "춘천") &
    (model_df["heavy"] == 1)
).values

print("춘천 호우일 데이터:", heavy_mask.sum(), "건")

s_heavy = plot_shap(
    heavy_mask,
    "춘천 호우일 변수 중요도",
    "07_shap_chuncheon_heavy.png"
)

# ③ 춘천 야외레저 예측 vs 실제 매출

In [ ]:
# ============================================================
# ③-1. 춘천 야외레저 실제 매출 vs 예측
#      2025.08 ~ 2025.12 전체 시험 기간
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. 전체 시험 결과에서 춘천 야외레저 추출
# ============================================================

d = test[(test["region"] == "춘천")
    & (test["ind_grp"] == "야외레저")
    & (test["date"] < "2025-11-01")].sort_values("date").copy()

print("춘천 야외레저 데이터:", len(d), "건")
print("호우일:", d["heavy"].sum(), "건")


# ============================================================
# 2. 날짜 형식 확인
# ============================================================

d["date"] = pd.to_datetime(d["date"])


# ============================================================
# 3. 실제 매출 vs 날씨 없는 모델 vs 날씨 포함 모델
# ============================================================

fig, ax = plt.subplots(figsize=(14, 5))

# 실제 매출
ax.plot(
    d["date"],
    d["sales"] / 1e8,
    color="black",
    lw=1.8,
    label="실제 매출"
)

# ② 날씨 없는 모델
ax.plot(
    d["date"],
    d["pred_lgbm_no_weather"] / 1e8,
    color="#8fb3d9",
    lw=1.2,
    label="② 날씨 없음"
)

# ③ 날씨 포함 모델
ax.plot(
    d["date"],
    d["pred_lgbm_weather"] / 1e8,
    color="#1f5fa8",
    lw=1.2,
    label="③ 날씨 포함"
)


# ============================================================
# 4. 호우일을 빨간색 배경으로 표시
# ============================================================

heavy_days = d[d["heavy"] == 1]

for i, t in enumerate(heavy_days["date"]):

    ax.axvspan(
        t - pd.Timedelta(hours=12),
        t + pd.Timedelta(hours=12),
        color="red",
        alpha=0.18,
        label="호우일" if i == 0 else None
    )


# ============================================================
# 5. 그래프 설정
# ============================================================

ax.set_xlabel("날짜")
ax.set_ylabel("일 매출 (억 원)")

ax.set_title(
    "춘천 야외레저: 실제 매출 vs 예측 "
)

ax.legend()
ax.grid(alpha=0.3)

plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig(BASE / "outputs/figures/07_chuncheon_outdoor_pred.png", dpi=200, bbox_inches="tight")
plt.show()


# ============================================================
# 6. 호우일 데이터 확인


# ============================================================
# 6. 호우일 데이터 확인
# ============================================================

h = d[
    d["heavy"] == 1
][[
    "date",
    "region",
    "ind_grp",
    "sales",
    "pred_lgbm_no_weather",
    "pred_lgbm_weather"
]].copy()


# ============================================================
# 7. rain(강수량)은 model_df에서 가져오기
# ============================================================

rain_info = (
    model_df[
        ["date", "region", "ind_grp", "rain"]
    ]
    .drop_duplicates(
        ["date", "region", "ind_grp"]
    )
)


h = h.merge(
    rain_info,
    on=["date", "region", "ind_grp"],
    how="left"
)


# ============================================================
# 8. 예측 오차 계산
# ============================================================

h["②오차%"] = (
    (h["pred_lgbm_no_weather"] / h["sales"] - 1) * 100
).round(1)

h["③오차%"] = (
    (h["pred_lgbm_weather"] / h["sales"] - 1) * 100
).round(1)


# ============================================================
# 9. 보기 좋은 순서로 정리
# ============================================================

h = h[
    [
        "date",
        "rain",
        "sales",
        "pred_lgbm_no_weather",
        "pred_lgbm_weather",
        "②오차%",
        "③오차%"
    ]
]


# ============================================================
# 10. 결과 출력
# ============================================================

print("\n======================================")
print("===== 춘천 야외레저 호우일 분석 =====")
print("======================================")

print("전체 시험 데이터:", len(d), "건")
print("호우일:", len(h), "건")
print()

print(h.to_string(index=False))